# Разбор датасета

Выполняется **до** `model.ipynb`. Здесь мы смотрим, из чего состоит датасет, находим копии
одних и тех же снимков и ошибки разметки, чистим и решаем, какие снимки идут в train и test.
Результат — манифест `_work/manifest.csv` (по строке на снимок): готовый набор, в котором каждый
снимок встречается один раз. `model.ipynb` читает только его — остаются аугментация и обучение.

Код — в `data_prep/`; настройки (пороги, `KEEP_PRIORITY`) — в `mri/config.py`.
Манифест одинаков при каждом запуске: после изменения датасета или настроек выполните ноутбук заново.
Файлы в `dataset/` не изменяются: исключённые снимки только помечаются в манифесте.

In [ ]:
import sys
from pathlib import Path

# Корень проекта — ближайшая папка, где лежит mri/: чтобы импортировались mri и data_prep.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "mri").is_dir())
sys.path.insert(0, str(ROOT))

import cv2
import numpy as np
import pandas as pd
from IPython.display import Image, display

from data_prep.split import build_manifest, save_manifest
from mri.config import KEEP_PRIORITY, NEAR_DUP_SIZE, NEAR_DUP_THRESHOLD, SAME_IMAGE_THRESHOLD, SOURCES
from mri.preprocess import prepare

pd.set_option("display.width", 160)


def show(pairs, scale=0.5):
    """Пары снимков рядом (после предобработки): список (файл, файл)."""
    rows = [np.hstack([prepare(ROOT / a), prepare(ROOT / b)]) for a, b in pairs]
    grid = cv2.resize(np.vstack(rows), None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    display(Image(cv2.imencode(".png", grid)[1].tobytes()))


print("Источники:", list(SOURCES))
print(f"Копии {NEAR_DUP_SIZE}×{NEAR_DUP_SIZE}; группа > {NEAR_DUP_THRESHOLD}, тот же снимок > {SAME_IMAGE_THRESHOLD}")
print("Какую копию оставлять:", KEEP_PRIORITY)

## 1. Поиск похожих снимков

SHA-256 находит только побайтные копии, а один и тот же снимок в разных источниках обычно
пересохранён. Поэтому каждый снимок уменьшается до 32×32 и сравнивается со всеми остальными
как есть, отражённым и повёрнутым на 180°; похожесть — корреляция (1 — одинаковые).

Два уровня, пороги проверены глазами:
- **image** (> 0.98) — один и тот же снимок: пересохранённый, отражённый, обрезанный иначе, с шумом
  или другим контрастом. От него остаётся один файл.
- **group** (> 0.97, по цепочке) — тот же снимок или соседние срезы той же серии, то есть, скорее
  всего, тот же пациент. Группа целиком идёт в одну часть: train или test.

Здесь же выполняются чистка и разбиение (разделы 5–6). Занимает около минуты.

In [ ]:
m = build_manifest()
m.head()

## 2. Состав

In [ ]:
print(m.pivot_table(index="source", columns="original_split", values="file", aggfunc="count", margins=True,
                     margins_name="всего"))
m.pivot_table(index="source", columns="class", values="file", aggfunc="count", margins=True, margins_name="всего")

## 3. Копии одних и тех же снимков

Сколько на самом деле разных снимков и насколько источники пересекаются друг с другом.
Ниже — по паре файлов из нескольких случайных копий: слева и справа один и тот же снимок.

In [ ]:
copies = m.groupby("image").size()
print(f"Файлов: {len(m)}, разных снимков: {len(copies)} ({len(copies) / len(m):.0%}), "
      f"групп: {m['group'].nunique()}")
print("Сколько файлов у одного снимка (файлов: сколько снимков):", copies.value_counts().sort_index().to_dict())

in_other_source = m.groupby("image")["source"].transform("nunique") > 1
print("\nДоля файлов, у которых тот же снимок есть в другом источнике:")
print(in_other_source.groupby(m["source"]).mean().round(3).to_string())

# Какие источники с какими пересекаются: число снимков, которые есть в обоих.
present = pd.crosstab(m["image"], m["source"]) > 0
present.T.astype(int) @ present.astype(int)

In [ ]:
multi = copies[copies > 1].index.to_numpy()
picked = np.random.default_rng(0).choice(multi, size=6, replace=False)
show([tuple(m.loc[m["image"] == g, "file"].iloc[:2]) for g in picked])

## 4. Ошибки разметки

Группы, где один и тот же снимок в разных местах помечен разными классами. Какой класс верный,
неизвестно, поэтому такие группы исключаются целиком (`reason = label_conflict`).
Ниже — по одному снимку каждого из двух классов из каждой такой группы.

In [ ]:
conflicts = m[m["reason"] == "label_conflict"].sort_values(["group", "class"])
print(f"Групп с разными классами: {conflicts['group'].nunique()}, файлов в них: {len(conflicts)}")
print(conflicts.groupby("group")["class"].agg(lambda c: ", ".join(sorted(set(c)))).value_counts().to_string())
show([tuple(g.drop_duplicates("class")["file"].iloc[:2]) for _, g in conflicts.groupby("group")])

## 5. Чистка

- **duplicate** — лишняя копия снимка; остаётся один файл (`duplicate_of`): без `aug` в имени,
  затем по источнику в порядке `KEEP_PRIORITY` (BDNeuro последним — в нём авторская аугментация).
- **seen_in_training** — тестовый снимок, чья группа есть в Training: модель видела этот снимок
  или соседний срез, честно оценить на нём нельзя.
- **label_conflict** — раздел 4.

Ниже — пары «исключённый файл → оставленный».

In [ ]:
print(m.pivot_table(index="reason", columns="original_split", values="file", aggfunc="count", fill_value=0)
      .rename(index={"": "(остаются)"}))
print()
print(m.pivot_table(index="source", columns="reason", values="file", aggfunc="count", fill_value=0)
      .rename(columns={"": "(остаются)"}))
dup = m[m["reason"] == "duplicate"].sample(6, random_state=0)
show(list(zip(dup["file"], dup["duplicate_of"])))

## 6. Итоговый набор и сохранение

Training → train, Testing → test. Проверки: группы train и test не пересекаются, от каждого
снимка остался один файл.

In [ ]:
used = m[m["split"] != "excluded"]
assert (used.groupby("group")["split"].nunique() == 1).all(), "группа попала в две части"
assert (used.groupby("image").size() == 1).all(), "у снимка осталось больше одного файла"

print(used.pivot_table(index="source", columns="split", values="file", aggfunc="count", margins=True,
                       margins_name="всего")[["train", "test", "всего"]])
print()
print(used.pivot_table(index="class", columns="split", values="file", aggfunc="count")[["train", "test"]])
save_manifest(m)